# Data Leakage in Machine Learning

Data leakage is one of the most dangerous and misunderstood problems in machine learning. It rarely raises an error. Instead, it quietly produces excellent validation scores and then fails in the real world.

## What Is Data Leakage?

Data leakage occurs when a model uses information during training that would not be available at prediction time in the real world. In other words, the model is learning from future data, target-derived signals, or improperly shared information between training and test sets.

The core rule is simple: a model must only learn from information that would exist at the moment a real-world prediction is made.

## Why Data Leakage Is Dangerous

Leakage is dangerous because it produces artificially high performance metrics, hides generalization failure, misleads stakeholders, wastes engineering effort, and damages trust when deployed systems fail. The most dangerous part is that leakage often looks like success.

## Common Leakage Patterns

### 1. Target Leakage
A feature is derived from or dependent on the target. Example: using `CancellationReason` to predict churn when that field only exists after churn has already happened.

### 2. Temporal Leakage
A feature includes future information. Example: using data recorded after the prediction decision point, such as post-discharge information in a readmission model.

### 3. Train-Test Contamination
A preprocessing step or aggregation is fit on the full dataset before splitting. Example: scaling all rows before train-test split or computing global category statistics from the entire dataset.

### 4. Target-Derived Features
A feature directly encodes the answer. Example: `DaysUntilDefault` for a default prediction task.

## The Prediction Moment Test

The most reliable way to catch leakage is to simulate the exact moment of prediction and ask: what information would actually exist right then? If a feature does not exist at prediction time, it should not be used.

## Prevention Checklist

- Separate `X` and `y` early.
- Split into train and test before fitting anything.
- Fit preprocessing only on training data.
- Avoid features derived from the target.
- Avoid future information.
- Review every feature using the prediction moment test.
- Document feature availability assumptions.
- Check suspiciously strong correlations carefully.

## Practical Example

The code below shows the difference between an incorrect preprocessing workflow and a correct one. The key idea is that anything that fits, learns statistics, or aggregates information must only be exposed to training data.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = pd.DataFrame({
    'age': [22, 25, 31, 40, 45, 52, 28, 36],
    'income': [30, 35, 50, 70, 80, 95, 40, 60],
    'churn': [0, 0, 0, 1, 1, 1, 0, 1]
})

X = data[['age', 'income']]
y = data['churn']

# Incorrect: fitting on the full dataset leaks test distribution information
leaky_scaler = StandardScaler()
X_leaky = leaky_scaler.fit_transform(X)

X_train_leaky, X_test_leaky, y_train, y_test = train_test_split(
    X_leaky, y, test_size=0.25, random_state=42
)

# Correct: split first, then fit only on training data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

safe_scaler = StandardScaler()
X_train_scaled = safe_scaler.fit_transform(X_train)
X_test_scaled = safe_scaler.transform(X_test)

print('Leaky train shape:', X_train_leaky.shape)
print('Safe train shape:', X_train_scaled.shape)
print('Safe test shape:', X_test_scaled.shape)

## Closing Reflection

Data leakage is not a coding error. It is a thinking error. If the model sees information it would not have in real life, the evaluation is invalid. Guard the boundary, protect the test set, and build honest models.

### Bonus Resources

- Kaggle Tutorial on Data Leakage
- Google Machine Learning Crash Course - Data Preparation